# Predicting 30-Day Hospital Readmission Among Patients With Diabetes

**HSE 751: Programming for Health Data Science**  
**Final Project Pipeline** (Week 4 Final Project Checkpoint; builds on the Week 3 Dataset Selection and Final Project Planning Lab)

## 1. Data Selection

### Dataset

**Dataset:** Diabetes 130-US Hospitals for Years 1999–2008  
**Source:** UCI Machine Learning Repository  
**Source URL:** https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008  
**Project data file:** `diabetic_data.csv`, the original file distributed by UCI (dataset ID 296), stored in this project's GitHub repository

### Dataset Selection and Justification

For my final project, I selected the **Diabetes 130-US Hospitals for Years 1999–2008** dataset from the UCI Machine Learning Repository. The dataset contains over 100,000 hospital encounters for patients with diabetes and includes demographic, clinical, medication, and healthcare utilization information.

I selected this dataset because it supports a clinically meaningful prediction problem: identifying hospital encounters at risk of readmission within 30 days. Early hospital readmission is an important clinical outcome and provides an opportunity to examine how multiple types of patient-level healthcare data can contribute to risk prediction. The project also aligns with my interests in clinical outcomes research, healthcare utilization, and the application of data science to patient-level clinical data.

Beyond the specific prediction problem, I plan to use this dataset as the initial demonstration of a reproducible clinical outcomes analytics pipeline. The longer-term goal is to organize the workflow so that data validation, preprocessing, statistical analysis, model training, and model evaluation can eventually be adapted to other compatible patient-level clinical datasets.

### Project Criteria

This dataset satisfies the requirements for the final project:

- **Binary classification:** The original readmission outcome will be transformed into a binary target indicating readmission within 30 days versus no readmission within 30 days.
- **Minimum of 300 observations:** The dataset contains more than 100,000 hospital encounters.
- **Minimum of 10 predictor variables:** The dataset contains substantially more than 10 candidate demographic, clinical, medication, and healthcare utilization variables.
- **Relevant subject area:** The dataset supports a clinically meaningful health outcomes prediction problem that aligns with my academic and research interests.

## 2. Data Import and Validation

The dataset is loaded from `diabetic_data.csv`, the original file distributed by the UCI Machine Learning Repository (dataset ID 296) and stored in this project's GitHub repository. It contains the same 101,766 encounters and the same 47 predictor variables as the `ucimlrepo` version used in Week 3. It also retains two identifier columns, `encounter_id` and `patient_nbr`, which `ucimlrepo` omits. `patient_nbr` is needed so that the same patient does not appear in both the training and test data.

The notebook loads the file from the Colab session if it has been uploaded, and otherwise directly from GitHub, so it can be run from start to finish without manual file handling.

### Enhancement 1: Data Contract and Automated Validation

**What was modified:** All dataset-specific expectations (data source, expected columns, target values, allowed category levels, plausible numeric ranges, and missing-value codes) were moved into a single `DATA_CONTRACT` dictionary. Two reusable functions were added: `load_data()` and `validate_data()`, plus `report_validation()` to display the results.

**Why it was implemented:** In Week 3, data loading and checks were hard-coded throughout the notebook. Any problem in the data, such as a renamed column or an unexpected category, would only show up later as a confusing error in a downstream step.

**How it improves the workflow:** The pipeline now checks the data immediately after import and stops with a clear message if critical expectations are not met, while non-critical issues are reported as warnings. Because the expectations live in one place, adapting the pipeline to another compatible clinical dataset requires editing the contract rather than the analysis code.

In [ ]:
# Import packages used throughout the pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
# =============================================================================
# DATA CONTRACT
# All dataset-specific expectations live here. To adapt the pipeline to another
# compatible dataset, edit this dictionary rather than the analysis code.
# =============================================================================
DATA_CONTRACT = {
    # Data source: a local copy in the Colab session is used first; the GitHub
    # copy is used as a fallback.
    "local_path": "diabetic_data.csv",
    "remote_url": "https://raw.githubusercontent.com/audreybyrne27/HSE751-Final-Project/main/diabetic_data.csv",

    # Structure
    "expected_n_columns": 50,
    "id_columns": ["encounter_id", "patient_nbr"],

    # Outcome
    "target_column": "readmitted",
    "target_allowed_values": {"NO", ">30", "<30"},
    "positive_class_value": "<30",

    # In the raw UCI file, "?" means missing. The text "None" in the two lab
    # columns means "test not performed" and is kept as a real category.
    "missing_value_codes": ["?"],

    # Plausible ranges for numeric variables (values outside trigger a warning)
    "numeric_ranges": {
        "time_in_hospital":   (1, 14),
        "num_lab_procedures": (0, 200),
        "num_procedures":     (0, 10),
        "num_medications":    (0, 100),
        "number_outpatient":  (0, 100),
        "number_emergency":   (0, 100),
        "number_inpatient":   (0, 100),
        "number_diagnoses":   (1, 20),
    },

    # Integer-coded variables that represent categories, not quantities
    "coded_categorical_columns": [
        "admission_type_id", "discharge_disposition_id", "admission_source_id"
    ],

    # Allowed levels for key categorical variables
    "categorical_allowed_values": {
        "gender": {"Female", "Male", "Unknown/Invalid"},
        "age": {"[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
                "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"},
        "max_glu_serum": {"None", "Norm", ">200", ">300"},
        "A1Cresult": {"None", "Norm", ">7", ">8"},
        "change": {"No", "Ch"},
        "diabetesMed": {"No", "Yes"},
    },
}

In [ ]:
def load_data(contract):
    """Load the dataset from a local file if present, otherwise from GitHub."""
    local_path = Path(contract["local_path"])
    read_kwargs = dict(
        na_values=contract["missing_value_codes"],  # convert "?" to NaN
        keep_default_na=False,                      # keep the text "None" as a category
        low_memory=False,                           # avoids the mixed-type warning
    )
    source = str(local_path) if local_path.exists() else contract["remote_url"]

    try:
        df = pd.read_csv(source, **read_kwargs)
    except Exception as error:
        raise FileNotFoundError(
            f"Could not load the dataset from '{source}'. Upload diabetic_data.csv "
            f"to the Colab session or check the GitHub URL in DATA_CONTRACT.\n{error}"
        )

    print(f"Loaded dataset from: {source}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df


raw_df = load_data(DATA_CONTRACT)
raw_df.head()

In [ ]:
def validate_data(df, contract):
    """Check the dataset against the data contract.
    Returns a dictionary of critical errors and non-critical warnings."""
    errors, warnings = [], []

    # 1. Structure
    if df.shape[1] != contract["expected_n_columns"]:
        errors.append(f"Expected {contract['expected_n_columns']} columns, found {df.shape[1]}.")

    required = (contract["id_columns"] + [contract["target_column"]]
                + list(contract["numeric_ranges"]) + contract["coded_categorical_columns"]
                + list(contract["categorical_allowed_values"]))
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        errors.append(f"Missing required columns: {missing_cols}")
        return {"errors": errors, "warnings": warnings}  # cannot continue checking

    # 2. Each encounter should appear once
    if df["encounter_id"].duplicated().any():
        errors.append(f"{df['encounter_id'].duplicated().sum()} duplicate encounter_id values.")

    # 3. Target values
    target = df[contract["target_column"]]
    unexpected = set(target.dropna().unique()) - contract["target_allowed_values"]
    if unexpected:
        errors.append(f"Unexpected target values: {unexpected}")
    if target.isna().any():
        errors.append("Target column contains missing values.")

    # 4. Numeric type and plausible range
    for col, (low, high) in contract["numeric_ranges"].items():
        if not pd.api.types.is_numeric_dtype(df[col]):
            errors.append(f"'{col}' should be numeric but is {df[col].dtype}.")
            continue
        n_out = ((df[col] < low) | (df[col] > high)).sum()
        if n_out:
            warnings.append(f"'{col}': {n_out} values outside expected range [{low}, {high}].")

    # 5. Categorical levels
    for col, allowed in contract["categorical_allowed_values"].items():
        unexpected = set(df[col].dropna().unique()) - allowed
        if unexpected:
            errors.append(f"'{col}' contains unexpected values: {unexpected}")

    # 6. Repeated patients (informational; handled later by splitting on patient)
    n_patients = df["patient_nbr"].nunique()
    if n_patients < len(df):
        warnings.append(f"{len(df):,} encounters come from {n_patients:,} unique patients "
                        f"({len(df) - n_patients:,} repeat encounters). "
                        f"Train/test split must be done by patient.")

    return {"errors": errors, "warnings": warnings}


def report_validation(results):
    """Print the validation report and stop the pipeline if critical errors exist."""
    print("DATA VALIDATION REPORT")
    print("=" * 60)
    for w in results["warnings"]:
        print(f"  WARNING: {w}")
    for e in results["errors"]:
        print(f"  ERROR:   {e}")
    if results["errors"]:
        raise ValueError(f"Validation failed with {len(results['errors'])} error(s). See report above.")
    print(f"  PASSED: no critical errors ({len(results['warnings'])} warning(s) to review).")


report_validation(validate_data(raw_df, DATA_CONTRACT))

In [ ]:
# Separate the 47 predictor variables from the original outcome.
# Identifier columns are kept in raw_df (needed later for the patient-level split)
# but excluded from the predictors.
ID_COLUMNS = DATA_CONTRACT["id_columns"]
TARGET_COLUMN = DATA_CONTRACT["target_column"]

X = raw_df.drop(columns=ID_COLUMNS + [TARGET_COLUMN]).copy()
y = raw_df[[TARGET_COLUMN]].copy()

print(f"Predictor data shape: {X.shape}")
print(f"Target data shape: {y.shape}")

## 3. Dataset Structure

The imported dataset contains **101,766 hospital encounters**, **47 predictor variables**, and **1 target variable**. Before defining the final binary prediction outcome or performing preprocessing, the raw dataset structure is examined to understand the available variables, data types, and example observations.

In Week 3, the `ucimlrepo` import generated a mixed-data-type warning for one column. Loading the raw CSV with `low_memory=False` and declaring `"?"` as the missing-value code resolves this, so all count variables are read as integers and the validation step confirms their types.

In [ ]:
# Display the first five rows of the predictor dataset
X.head()

In [ ]:
# Display the names of all predictor variables
print("Predictor variables:")
for i, column in enumerate(X.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
# Examine the structure and data types of the predictor dataset
X.info()

In [ ]:
# Examine the original target variable supplied by UCI
print("Target variable name:")
print(y.columns.tolist())

print("\nFirst five target values:")
display(y.head())

print("\nTarget value counts:")
print(y.iloc[:, 0].value_counts(dropna=False))

## 4. Target Variable

The original target variable provided by UCI is `readmitted`. It contains three categories:

- `NO` = no recorded readmission
- `>30` = readmission more than 30 days after discharge
- `<30` = readmission within 30 days of discharge

For this project, the prediction problem is **binary classification of 30-day hospital readmission**. The original target is therefore transformed into:

\[
Y_i =
\begin{cases}
1, & \text{if patient encounter } i \text{ has readmitted = <30} \\
0, & \text{if patient encounter } i \text{ has readmitted = >30 or NO}
\end{cases}
\]

The resulting target variable, `readmitted_30d`, represents whether an encounter was followed by a readmission within 30 days.

This outcome definition focuses the classification problem on early readmission. Encounters with readmission after 30 days are included in the negative class because they did not experience the specific outcome being predicted within the 30-day window.

In [ ]:
# Create the binary 30-day readmission target.
# 1 = readmitted within 30 days
# 0 = not readmitted within 30 days

readmitted_30d = (y["readmitted"] == "<30").astype(int)

# Examine the new binary target
print("Binary target value counts:")
print(readmitted_30d.value_counts().sort_index())

print("\nBinary target proportions:")
print(readmitted_30d.value_counts(normalize=True).sort_index().round(4))

In [ ]:
# Validate the binary target
assert len(readmitted_30d) == len(X), "Target and predictor row counts do not match."
assert set(readmitted_30d.unique()) == {0, 1}, "Target must contain only 0 and 1."
assert readmitted_30d.sum() == (y["readmitted"] == "<30").sum(), \
    "Binary target does not match the original <30 readmission category."

print("Binary target validation passed.")

### Target Distribution

The binary outcome contains **11,357 encounters (11.16%)** with a readmission within 30 days and **90,409 encounters (88.84%)** without a readmission within 30 days.

This demonstrates substantial **class imbalance**, with approximately one in nine encounters belonging to the positive class. A classifier that simply predicted no 30-day readmission for every encounter would therefore achieve high overall accuracy despite having no ability to identify patients who experience the outcome of interest.

This imbalance will be important when selecting model evaluation metrics. For the final classification pipeline, **F1 score is proposed as the primary evaluation metric** because it combines precision and recall and therefore evaluates performance in identifying the minority positive class without relying on accuracy alone. Additional metrics can be used later to provide a more complete assessment of model performance.

## 5. Predictor Variables

The dataset contains **47 candidate predictor variables** representing multiple aspects of the patient and hospital encounter. These variables include both numerical and categorical data and can be organized into several clinically meaningful domains.

### Demographic Characteristics

- `race`
- `gender`
- `age`
- `weight`

These variables describe basic patient demographic and anthropometric characteristics.

### Hospital Encounter Characteristics

- `admission_type_id`
- `discharge_disposition_id`
- `admission_source_id`
- `time_in_hospital`
- `payer_code`
- `medical_specialty`

These variables describe characteristics of the hospitalization, including the type and source of admission, discharge disposition, length of stay, payer, and admitting medical specialty.

### Clinical Care and Healthcare Utilization

- `num_lab_procedures`
- `num_procedures`
- `num_medications`
- `number_outpatient`
- `number_emergency`
- `number_inpatient`
- `number_diagnoses`

These variables quantify care delivered during the encounter and prior healthcare utilization.

### Diagnoses

- `diag_1`
- `diag_2`
- `diag_3`

These variables represent the primary and additional diagnosis codes associated with the encounter.

### Laboratory Results

- `max_glu_serum`
- `A1Cresult`

These variables provide available information about glucose and hemoglobin A1C testing.

### Diabetes Medications

The dataset contains multiple variables describing the status of individual diabetes medications, including:

`metformin`, `repaglinide`, `nateglinide`, `chlorpropamide`, `glimepiride`, `acetohexamide`, `glipizide`, `glyburide`, `tolbutamide`, `pioglitazone`, `rosiglitazone`, `acarbose`, `miglitol`, `troglitazone`, `tolazamide`, `examide`, `citoglipton`, `insulin`, `glyburide-metformin`, `glipizide-metformin`, `glimepiride-pioglitazone`, `metformin-rosiglitazone`, and `metformin-pioglitazone`.

### Overall Diabetes Treatment

- `change`
- `diabetesMed`

These variables indicate whether the diabetes medication regimen changed during the encounter and whether diabetes medication was prescribed.

Together, these predictors provide demographic, clinical, utilization, laboratory, and treatment information that may be evaluated for their relationship with 30-day hospital readmission.

## 6. Descriptive Statistics

Descriptive statistics were calculated to characterize the hospital encounters before further exploratory analysis. Because the dataset contains both numerical and categorical predictors, the variables are summarized using methods appropriate to their data types.

For numerical variables, the analysis reports the count, mean, standard deviation, median, interquartile range, minimum, and maximum. For selected categorical variables, frequency distributions are examined to understand the composition of the study population and identify potentially sparse or unusual categories.

In [ ]:
# Select numerical variables that represent counts or quantities.
numeric_variables = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

# Calculate descriptive statistics.
numeric_summary = X[numeric_variables].describe().T

# Add the median and interquartile range (IQR).
numeric_summary["median"] = X[numeric_variables].median()
numeric_summary["IQR"] = (
    X[numeric_variables].quantile(0.75)
    - X[numeric_variables].quantile(0.25)
)

# Select and organize the statistics for display.
numeric_summary = numeric_summary[
    ["count", "mean", "std", "min", "25%", "median", "75%", "IQR", "max"]
]

numeric_summary.round(2)

In [ ]:
# Examine frequency distributions for selected categorical predictors.
categorical_variables = [
    "race",
    "gender",
    "age",
    "max_glu_serum",
    "A1Cresult",
    "insulin",
    "change",
    "diabetesMed"
]

for variable in categorical_variables:
    print(f"\n{variable}")
    print("-" * 40)
    print(X[variable].value_counts(dropna=False))

In [ ]:
# Count the number of unique observed values in each predictor.
unique_counts = X.nunique(dropna=False).sort_values()

print("Number of unique values by predictor:")
print(unique_counts)

### Descriptive Statistics Interpretation

The dataset contains 101,766 hospital encounters with substantial variation in hospitalization characteristics and prior healthcare utilization. The median length of stay was **4 days** (IQR: 2–6), and encounters included a median of **44 laboratory procedures** (IQR: 31–57), **15 medications** (IQR: 10–20), and **8 recorded diagnoses** (IQR: 6–9).

Prior healthcare utilization was highly concentrated at zero. The median number of prior outpatient and emergency encounters was **0**, with an IQR of 0–0 for both variables. Prior inpatient utilization also had a median of 0, although its upper quartile was 1. The substantially higher maximum values for outpatient (42), emergency (76), and inpatient (21) utilization suggest right-skewed distributions and the presence of patients with substantially greater prior healthcare use.

The population was concentrated in older age groups, particularly ages **70–79** (n = 26,068), **60–69** (n = 22,483), and **80–89** (n = 17,197). The dataset contained 54,708 female and 47,055 male encounters, with three observations coded as `Unknown/Invalid`.

Several variables also show important patterns relevant to future preprocessing. `A1Cresult` and `max_glu_serum` are unavailable for most encounters, while the diagnosis variables (`diag_1`, `diag_2`, and `diag_3`) contain hundreds of distinct values. In contrast, `examide` and `citoglipton` each contain only one observed value and therefore provide no variation in this dataset. These characteristics will be considered during the data quality assessment and preprocessing plan.

In [ ]:
# Create an analysis dataset containing predictors and the binary outcome.
analysis_df = X.copy()
analysis_df["readmitted_30d"] = readmitted_30d

print("Analysis dataset shape:", analysis_df.shape)
analysis_df.head()

## 7. Exploratory Visualizations

### 30-Day Readmission by Age Group

The first visualization examines whether the proportion of encounters followed by a 30-day readmission differs across age groups. Because `age` is recorded as ordered age intervals rather than an exact continuous age, readmission rates are calculated separately for each available age category.

In [ ]:
# Define the clinical ordering of the age categories.
age_order = [
    "[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
    "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"
]

# Calculate 30-day readmission rate within each age group.
age_readmission = (
    analysis_df.groupby("age", observed=True)["readmitted_30d"]
    .agg(["mean", "count"])
    .reindex(age_order)
    .reset_index()
)

age_readmission["readmission_rate_percent"] = (
    age_readmission["mean"] * 100
)

age_readmission[["age", "count", "readmission_rate_percent"]].round(2)

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=age_readmission,
    x="age",
    y="readmission_rate_percent"
)

plt.title("30-Day Readmission Rate by Age Group")
plt.xlabel("Age Group (years)")
plt.ylabel("30-Day Readmission Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### Interpretation

Thirty-day readmission rates varied across age groups, although the relationship was not consistently linear. The lowest observed rates occurred among patients ages 0–9 (1.86%) and 10–19 (5.79%), while the highest rate occurred among patients ages 20–29 (14.24%). Among the more heavily represented adult age groups, readmission rates generally ranged from approximately 9.7% to 12.1%.

The age groups differ substantially in sample size. For example, the 0–9 age group contains only 161 encounters, compared with 26,068 encounters among patients ages 70–79. Therefore, apparent differences in the smaller age groups should be interpreted cautiously because their estimated readmission rates may be less stable.

Overall, age appears to have a potentially non-linear relationship with 30-day readmission in this dataset. This suggests that the ordered age categories may contain useful predictive information, but the exploratory results do not support assuming a simple linear increase in readmission risk with increasing age.

In [ ]:
# Create categories of prior inpatient utilization for visualization.
analysis_df["prior_inpatient_group"] = pd.cut(
    analysis_df["number_inpatient"],
    bins=[-1, 0, 1, 2, np.inf],
    labels=["0", "1", "2", "3+"]
)

# Calculate 30-day readmission rate by prior inpatient utilization.
inpatient_readmission = (
    analysis_df.groupby("prior_inpatient_group", observed=True)["readmitted_30d"]
    .agg(["mean", "count"])
    .reset_index()
)

inpatient_readmission["readmission_rate_percent"] = (
    inpatient_readmission["mean"] * 100
)

inpatient_readmission[
    ["prior_inpatient_group", "count", "readmission_rate_percent"]
].round(2)

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=inpatient_readmission,
    x="prior_inpatient_group",
    y="readmission_rate_percent"
)

plt.title("30-Day Readmission Rate by Prior Inpatient Utilization")
plt.xlabel("Number of Prior Inpatient Encounters")
plt.ylabel("30-Day Readmission Rate (%)")
plt.tight_layout()
plt.show()

#### Interpretation

Prior inpatient utilization showed a clear positive relationship with 30-day readmission. Among encounters with no prior inpatient admissions, the 30-day readmission rate was **8.44%**. The rate increased to **12.92%** among patients with one prior inpatient encounter, **17.43%** among those with two, and **25.66%** among those with three or more.

This represents a substantial gradient in the observed outcome. Patients with three or more prior inpatient encounters had a 30-day readmission rate approximately three times that observed among patients with no prior inpatient utilization.

These findings suggest that prior inpatient utilization may contain useful information for predicting subsequent 30-day readmission. However, this visualization represents an unadjusted association and does not establish that prior hospitalization itself causes readmission. Other differences in disease burden, patient complexity, or healthcare utilization may contribute to the observed relationship.

## 8. Data Quality Assessment

Before developing a classification model, the dataset was evaluated for missing values, unusual coding, low-variation predictors, and other characteristics that may require preprocessing. This assessment is intended to identify potential data-quality challenges without modifying the source dataset at this stage.

In [ ]:
# Calculate missingness for every predictor.
missing_summary = pd.DataFrame({
    "missing_count": X.isna().sum(),
    "missing_percent": (X.isna().mean() * 100)
})

# Retain variables containing at least one missing value.
missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_percent", ascending=False)
)

missing_summary.round(2)

In [ ]:
# Search categorical variables for common representations of
# missing, unknown, or invalid information.

object_columns = X.select_dtypes(include="object").columns

possible_missing_codes = [
    "?", "Unknown", "Unknown/Invalid",
    "Not Available", "None", "NULL"
]

for column in object_columns:
    observed_values = X[column].dropna().astype(str).unique()

    flagged_values = [
        value for value in observed_values
        if value in possible_missing_codes
    ]

    if flagged_values:
        print(f"{column}: {flagged_values}")

In [ ]:
# Identify predictors with no or very little variation.
unique_summary = pd.DataFrame({
    "unique_values": X.nunique(dropna=False),
    "most_common_percent": [
        X[col].value_counts(dropna=False, normalize=True).iloc[0] * 100
        for col in X.columns
    ]
})

unique_summary = unique_summary.sort_values(
    ["unique_values", "most_common_percent"],
    ascending=[True, False]
)

unique_summary.head(15).round(2)

In [ ]:
# Check for exact duplicate predictor rows.
duplicate_rows = X.duplicated().sum()

print(f"Exact duplicate predictor rows: {duplicate_rows}")

### Data Quality Findings

The data quality assessment identified several issues that will need to be addressed before model development.

**Missing data:** Missingness varies substantially across predictors. `weight` is missing for **96.86%** of encounters, and `medical_specialty` (49.08%) and `payer_code` (39.56%) also have substantial missingness. Missingness is considerably lower for `race` (2.23%) and the three diagnosis variables (0.02%–1.40%). Because the amount of missing information differs substantially across variables, a single preprocessing strategy is unlikely to be appropriate for every predictor.

**Laboratory tests not performed:** In the raw UCI file, `max_glu_serum` and `A1Cresult` use the value `None` to indicate that the test was not performed during the encounter, rather than leaving the value blank. These are therefore not missing data: serum glucose was not tested in **94.75%** of encounters and HbA1c was not tested in **83.28%**. Whether a test was ordered may itself carry clinical information, so `None` is retained as its own category.

**Improper or unusual coding:** The `gender` variable contains three observations coded as `Unknown/Invalid`. These values should be treated as invalid or unknown rather than as a meaningful gender category during preprocessing.

**Low-variation predictors:** `examide` and `citoglipton` each contain only one observed value and therefore provide no information for distinguishing between outcome classes. Several additional medication variables are near-constant, including `acetohexamide`, `glimepiride-pioglitazone`, `metformin-pioglitazone`, `metformin-rosiglitazone`, and `troglitazone`. Although these variables technically contain more than one observed value, nearly 100% of encounters fall into a single category.

**Duplicate observations and repeated patients:** No exact duplicate predictor rows were identified, and each `encounter_id` appears once. However, the 101,766 encounters come from only **71,518 unique patients** (`patient_nbr`), so many patients contribute more than one encounter. This will be handled by splitting the data into training and test sets at the patient level.

Overall, the dataset is sufficiently large and information-rich for the proposed classification problem, but substantial missingness, sparse predictors, categorical coding, and potential repeated encounters will require careful preprocessing before predictive modeling.

## 9. Anticipated Preprocessing and Potential Challenges

Several preprocessing steps will be considered before training the final classification models.

### Missing Data

Variables with extremely high missingness, such as `weight`, `max_glu_serum`, and `A1Cresult`, will require careful evaluation. Variables with very little observed information may be excluded from the predictive feature set if they cannot be used reliably. For variables with more moderate missingness, missingness may itself contain clinically relevant information; therefore, categorical missing or unknown levels and missingness indicators will be considered rather than automatically deleting incomplete encounters.

### Low-Variation Features

Constant predictors such as `examide` and `citoglipton` will be removed because they contain no discriminatory information. Near-constant medication variables will also be evaluated and may be removed if their extremely limited variation is unlikely to contribute useful predictive information.

### Categorical Variables

The dataset contains numerous categorical variables that cannot be entered directly into many machine-learning algorithms. Variables such as race, gender, age group, medication status, and other categorical characteristics will require appropriate encoding. Integer-coded variables such as `admission_type_id`, `discharge_disposition_id`, and `admission_source_id` will be treated as categorical identifiers rather than continuous numerical measurements.

### High-Cardinality Variables

The diagnosis variables (`diag_1`, `diag_2`, and `diag_3`) contain hundreds of distinct diagnosis codes, and `medical_specialty` also contains many categories. Direct one-hot encoding of every observed value could create a large and sparse feature space. These variables may require clinically meaningful grouping or other dimensionality-reduction strategies before modeling.

### Outcome Imbalance

Only **11.16%** of encounters experienced a readmission within 30 days. Because of this class imbalance, overall accuracy would provide an incomplete assessment of model performance. The project will use **F1 score as the primary evaluation metric**, with additional classification metrics reported to characterize model performance more completely.

### Potential Outcome Leakage and Cohort Definition

Predictors will be reviewed to ensure that they represent information that would reasonably be available at the intended time of prediction and do not directly reveal the future readmission outcome. Variables related to discharge disposition require particular attention because some discharge outcomes may affect whether a subsequent hospital readmission is possible or clinically meaningful.

### Repeated Patient Encounters

The source data are organized at the hospital-encounter level, so multiple encounters may potentially correspond to the same patient. For the final predictive pipeline, patient-level identifiers and the original dataset structure will be investigated so that repeated encounters can be handled appropriately and information from the same patient does not inadvertently appear in both training and test data.

### Reproducible Preprocessing

For the final project, preprocessing steps will be incorporated into a reproducible analytical pipeline. The planned workflow will define required variables and data types, validate incoming data, apply preprocessing consistently, train classification models, and evaluate performance. Where feasible, dataset-specific definitions will be separated from reusable pipeline components so that the framework can later be adapted to other compatible patient-level clinical datasets.

## 10. Project Planning

### Proposed Prediction Problem

The final project will develop a supervised binary classification model to predict **30-day hospital readmission among patients with diabetes**. The model will use demographic, clinical, medication, and healthcare utilization information available from the hospital encounter to identify encounters at increased risk of readmission within 30 days.

### Target Variable

The binary target variable is `readmitted_30d`, derived from the original UCI `readmitted` variable:

- `1` = readmitted within 30 days (`<30`)
- `0` = not readmitted within 30 days (`>30` or `NO`)

The resulting outcome is imbalanced: **11,357 encounters (11.16%)** experienced a 30-day readmission, compared with **90,409 encounters (88.84%)** without a 30-day readmission.

### Primary Evaluation Metric

The proposed primary evaluation metric is the **F1 score**:

$$
F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}
{\text{Precision} + \text{Recall}}
$$

F1 score is appropriate for this prediction problem because the positive class—30-day readmission—represents only 11.16% of encounters. Overall accuracy could appear high even for a model that performs poorly at identifying patients who experience a 30-day readmission.

F1 balances **precision**, the proportion of predicted readmissions that are true readmissions, and **recall**, the proportion of actual 30-day readmissions correctly identified by the model. Additional metrics, including precision, recall, confusion matrix, and area under the receiver operating characteristic curve (AUROC), may be reported as secondary measures to provide a more complete assessment of model performance.

### Final Project Direction

The diabetes readmission problem will serve as the initial implementation of a broader **reproducible clinical outcomes analytics pipeline**. The planned workflow will use a standardized data-contract approach to validate input data, preprocess predictors, perform exploratory and statistical analyses, train classification models, and evaluate model performance.

Where feasible, dataset-specific definitions such as the target and required predictors will be separated from reusable analytical components. This structure is intended to make the workflow reproducible for the current project while providing a foundation that could later be adapted to other compatible patient-level clinical outcomes datasets.